# 27 — MCP Sampling and Roots

Every primitive so far flows client → server: the client lists/calls
tools, reads resources, fetches prompts. **Sampling** and **roots** flip
that direction -- the *server* asks the *client* for something. Both
require a human in the loop per the spec's security guidance, and both are
**client** capabilities (declared by whoever embeds the MCP client, not by
the server).

## Sampling: servers borrowing the client's LLM

Sampling lets a tool implementation request an LLM completion *through the
client* instead of calling a model API directly -- no server-side API key
needed, and the client/user stays in control of which model actually runs
and whether the request is approved at all.

```json
// Server -> Client
{
  "jsonrpc": "2.0", "id": 1, "method": "sampling/createMessage",
  "params": {
    "messages": [{ "role": "user", "content": { "type": "text", "text": "What is the capital of France?" } }],
    "modelPreferences": { "hints": [{ "name": "claude-3-sonnet" }], "intelligencePriority": 0.8 },
    "maxTokens": 100
  }
}
```

`modelPreferences` is an abstraction, not a model name: `costPriority`,
`speedPriority`, and `intelligencePriority` (0-1 each) plus optional
`hints` the client may map to whatever equivalent model it actually has
access to. Per spec, the client **SHOULD** show the user the request (and
the generated response) for review before anything is sent back to the
server.

In [ ]:
# Our custom server's generate_poem tool calls ctx.session.create_message(...)
# internally (see examples/mcp_foundations/custom_server.py) -- which
# arrives at the client as a sampling/createMessage request. We fulfill it
# with a stand-in `sampling_callback` instead of a real model call, to keep
# this notebook free and offline; swap in a real Azure OpenAI call here to
# wire up genuine sampling.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

server_script = str(Path.cwd().parent / "examples" / "mcp_foundations" / "custom_server.py")
server_params = StdioServerParameters(command=sys.executable, args=[server_script])

async def handle_sampling_message(context, params):
    print("  [sampling request received]:", params.messages[0].content.text)
    return types.CreateMessageResult(
        role="assistant",
        content=types.TextContent(type="text", text="Roses are red,\nMCP flows through stdio."),
        model="stand-in-model",
        stopReason="endTurn",
    )

async def generate_poem(topic: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write, sampling_callback=handle_sampling_message) as session:
            await session.initialize()
            return await session.call_tool("generate_poem", arguments={"topic": topic})

poem = await generate_poem("stdio pipes")
print("\nfinal tool result:", poem.content[0].text)